# Journey A — Admin สร้างผู้ใช้ใหม่ (ยิง API ทีละขั้น พร้อมอีเมลจริง)

สเปกกำหนดว่าขั้นตอนนี้ **ไม่มีหน้าจอ** — แอดมินเรียก API ตรง ๆ notebook นี้จึงเดินแทนหน้าจอ
โดยยิงทีละ endpoint ให้เห็นทั้ง request และ response ตอนสาธิต

| ขั้น | ใครทำ | endpoint |
| --- | --- | --- |
| 1 | Admin | `POST /api/admin/invitations` — ออก token แล้ว **ส่งอีเมลคำเชิญ** |
| 2 | ผู้ถูกเชิญ | `GET /api/auth/invitation?token=` — ตรวจว่าลิงก์ยังใช้ได้ |
| 3 | ผู้ถูกเชิญ | `POST /api/auth/register` — ตั้งรหัสผ่าน แล้วระบบ **ส่ง OTP ทางอีเมล** |
| 4 | ผู้ถูกเชิญ | `POST /api/auth/verify-otp` — ยืนยันตัวตน ได้ session cookie |
| 5 | ผู้ถูกเชิญ | `GET /api/auth/me` — บัญชีเป็น `ACTIVE` แล้ว |

## ก่อนเริ่ม

- ต้องมี `requests` (`python3 -m pip install requests`) และเคอร์เนล Python ที่รัน notebook ได้
  เครื่องนี้ยังไม่มี jupyter — เปิดไฟล์นี้ใน VS Code (ต้องมี `ipykernel`) หรือ
  `python3 -m pip install jupyterlab ipykernel` แล้ว `jupyter lab`
- **อีเมลจะถูกส่งจริงก็ต่อเมื่อ checkout นั้นตั้ง `SMTP_USER` / `SMTP_PASS` ไว้แล้ว**
  ตอนนี้มีแค่ `main` ที่ตั้งไว้ — checkout อื่นจะพิมพ์อีเมลลง log แทน (เซลล์ที่ 2 บอกให้ว่าอยู่โหมดไหน)
- ใส่อีเมลของตัวเองในเซลล์ถัดไป ถ้าใช้ Gmail แนะนำ **plus-addressing**
  (`ชื่อคุณ+demo1@gmail.com`) จะได้เชิญซ้ำกี่รอบก็ได้โดยไม่ชนบัญชีเดิม เมลเข้ากล่องเดียวกัน

รายละเอียดของแต่ละหน้าจอที่คู่กับ API เหล่านี้อยู่ใน [`../docs/03-demo-walkthrough.md`](../docs/03-demo-walkthrough.md)

In [3]:
import json
from pathlib import Path
from urllib.parse import urlparse, parse_qs

import requests

# ── แก้ตรงนี้ ────────────────────────────────────────────────────────────
CHECKOUT = Path("/hdd1tb/bdi-project/main")     # checkout ที่จะยิง API ใส่ (main = ส่งอีเมลจริง)
API = "https://bdi-api.thammasorn.org"          # main · dev_01 http://localhost:4110
                                                #        dev_02 http://localhost:4120

INVITE_EMAIL = "thammasorn.han+demo2@gmail.com"
ROLE = "ORGANIZATION_USER"  # BDI_OFFICER · BDI_APPROVER · BDI_SPECIALIST
                            # ORGANIZATION_USER · ORGANIZATION_APPROVER
# ─────────────────────────────────────────────────────────────────────────

# main ตั้ง APP_URL เป็น https จึงออก session cookie แบบ Secure — ยิงผ่าน
# http://localhost:4000 จะล็อกอินผ่านแต่ทุก call ถัดไปได้ 401 เพราะ cookie ไม่ถูกส่งกลับ
# ใช้ URL https ข้างบนกับ main เสมอ


def env(name: str, default: str = "") -> str:
    """อ่านค่าจาก .env ของ checkout — ไฟล์นี้ไม่อยู่ใน git และเก็บ credential จริง"""
    for line in (CHECKOUT / ".env").read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        if key.strip() == name:
            return value.strip().strip('"').strip("'")
    return default


ADMIN_TOKEN = env("ADMIN_API_TOKEN")
APP_URL = env("APP_URL", "http://localhost:3000")
SMTP_USER = env("SMTP_USER")

http = requests.Session()  # เก็บ session cookie ให้อัตโนมัติหลังยืนยัน OTP


def call(method: str, path: str, **kwargs):
    """ยิง API แล้วพิมพ์ทั้งสถานะและ body ให้ดูสด ๆ ตอนสาธิต"""
    response = http.request(method, API + path, timeout=30, **kwargs)
    print(f"{method} {path}  →  {response.status_code} {response.reason}\n")
    try:
        body = response.json()
    except ValueError:
        print(response.text[:800])
        return response, None
    print(json.dumps(body, ensure_ascii=False, indent=2))
    return response, body


print("พร้อมแล้ว — ยิงไปที่", API)

พร้อมแล้ว — ยิงไปที่ https://bdi-api.thammasorn.org


## 0. ตรวจก่อนว่าระบบพร้อม และอีเมลจะถูกส่งจริงหรือไม่

In [4]:
call("GET", "/health/ready")

print()
if SMTP_USER:
    print(f"โหมดอีเมล : ส่งจริง ผ่าน {SMTP_USER}")
else:
    print("โหมดอีเมล : พิมพ์ลง log เท่านั้น (SMTP_USER ว่าง)")
    print("            ถ้าต้องการอีเมลจริง ตั้ง SMTP_USER/SMTP_PASS ใน .env แล้ว")
    print("            docker compose up -d backend  —  หรือชี้ CHECKOUT ไปที่ main")

print("ลิงก์ในอีเมลจะชี้ไปที่ :", APP_URL)
print("ADMIN_API_TOKEN       :", "อ่านจาก .env ได้แล้ว" if ADMIN_TOKEN else "ไม่พบใน .env")
print("อีเมลที่จะเชิญ         :", INVITE_EMAIL)

GET /health/ready  →  200 OK

{
  "status": "ok",
  "checks": {
    "database": {
      "status": "up"
    },
    "storage": {
      "status": "up"
    }
  }
}

โหมดอีเมล : ส่งจริง ผ่าน wanisara.harn@gmail.com
ลิงก์ในอีเมลจะชี้ไปที่ : https://bdi.thammasorn.org
ADMIN_API_TOKEN       : อ่านจาก .env ได้แล้ว
อีเมลที่จะเชิญ         : thammasorn.han+demo2@gmail.com


## 1. Admin ส่งคำเชิญ

ป้องกันด้วย `x-admin-token` ไม่ใช่ session เพราะผู้เรียกคือสคริปต์ของผู้ดูแลระบบ ไม่ใช่คนที่ล็อกอิน

**คำตอบจงใจไม่คืน token กลับมา** — token อยู่ในอีเมลเท่านั้น ถ้าเชิญอีเมลเดิมซ้ำ
คำเชิญอันเก่าจะถูก revoke อัตโนมัติ เหลือลิงก์ที่ใช้ได้อันเดียวเสมอ

In [5]:
response, invitation = call(
    "POST",
    "/api/admin/invitations",
    headers={"x-admin-token": ADMIN_TOKEN},
    json={"email": INVITE_EMAIL, "role": ROLE},
)

if response.status_code == 201:
    print("\n📧 เปิดกล่องจดหมายของ", INVITE_EMAIL, "— หัวข้อ 'คำเชิญเข้าใช้งาน Government Datahub Platform'")
elif response.status_code == 409:
    print("\nอีเมลนี้มีบัญชีที่ ACTIVE อยู่แล้ว — เปลี่ยนเป็น +demo2 แล้วรันใหม่")

POST /api/admin/invitations  →  201 Created

{
  "invitationId": "4d553628-2fe4-4f39-b024-35f1461260f9",
  "email": "thammasorn.han+demo2@gmail.com",
  "role": "ORGANIZATION_USER",
  "roleLabel": "ผู้ใช้จากหน่วยงาน",
  "expiresAt": "2026-08-11T14:18:13.814Z"
}

📧 เปิดกล่องจดหมายของ thammasorn.han+demo2@gmail.com — หัวข้อ 'คำเชิญเข้าใช้งาน Government Datahub Platform'


## 2. เปิดอีเมล แล้วเอาลิงก์มาวาง

ในอีเมลมีปุ่ม **เริ่มลงทะเบียน** ชี้ไปที่ `<APP_URL>/register?token=…`
คลิกขวาที่ปุ่ม → คัดลอกลิงก์ แล้ววางในเซลล์ถัดไป (จะวางทั้งลิงก์หรือเฉพาะ token ก็ได้)

> ถ้า checkout อยู่โหมดพิมพ์ลง log ให้ข้ามไปเซลล์ **หาลิงก์จาก log** ท้ายไฟล์นี้

In [ ]:
PASTED = ""  # ← วางลิงก์จากอีเมลตรงนี้

TOKEN = parse_qs(urlparse(PASTED).query).get("token", [PASTED])[0].strip()
print("token :", TOKEN[:12] + "…" if TOKEN else "(ยังไม่ได้วาง)")

call("GET", "/api/auth/invitation", params={"token": TOKEN})

## 3. ผู้ถูกเชิญกรอกข้อมูลและตั้งรหัสผ่าน

อีเมลมาจากคำเชิญ ไม่ได้ส่งขึ้นมา — กันคำเชิญถูกใช้ผิดคน

- เบอร์โทรต้องขึ้นต้น `0` และมี 9–10 หลัก
- รหัสผ่านอย่างน้อย 8 ตัว มีทั้งตัวอักษรและตัวเลข

ลองกรอกผิดดูได้ — ระบบตอบเป็น `{"error":"validation","fields":{…}}` ระบุทีละช่อง

In [ ]:
PROFILE = {
    "prefix": "นาย",
    "firstName": "ทดสอบ",
    "lastName": "ระบบดี",
    "phone": "0812345678",
    "password": "bdi12345",
}

response, _ = call("POST", "/api/auth/register", json={"token": TOKEN, **PROFILE})

if response.status_code == 202:
    print("\n📧 บัญชีถูกสร้างเป็น INVITED แล้ว — รหัส OTP 6 หลักกำลังไปที่กล่องจดหมายเดิม")

## 4. ยืนยันตัวตนด้วย OTP

รหัสอยู่ในหัวข้ออีเมล `รหัสยืนยันตัวตน 123456 — BDI Datahub` มีอายุ 10 นาที
กรอกผิดได้ 5 ครั้ง เกินนั้นต้องขอรหัสใหม่ (`POST /api/auth/resend-otp`)

ผ่านแล้วบัญชีเปลี่ยนเป็น `ACTIVE` และเซิร์ฟเวอร์ตั้ง session cookie ให้ —
`http` เก็บ cookie นั้นไว้ต่อ เซลล์ถัดไปจึงเรียก endpoint ที่ต้องล็อกอินได้เลย

In [ ]:
OTP = ""  # ← รหัส 6 หลักจากอีเมล

response, _ = call("POST", "/api/auth/verify-otp", json={"token": TOKEN, "code": OTP.strip()})

if response.status_code == 200:
    print("\ncookie ที่ได้รับ :", ", ".join(http.cookies.keys()) or "(ไม่มี)")

## 5. ตรวจผล

บัญชีใหม่เข้าใช้งานได้แล้ว — เข้า `APP_URL` ด้วยอีเมลนี้กับรหัสผ่านที่ตั้งไว้
เพื่อเดินต่อเป็น Journey B (สร้างหน่วยงาน) ได้ทันที

In [ ]:
call("GET", "/api/auth/me")

_, listing = call("GET", "/api/admin/invitations", headers={"x-admin-token": ADMIN_TOKEN})
if listing:
    mine = [i for i in listing["invitations"] if i["email"] == INVITE_EMAIL]
    print("\nคำเชิญของอีเมลนี้ (ล่าสุดอยู่บน) :")
    for item in mine:
        print(f"  {item['status']:9} {item['createdAt']}")
    print("\nACCEPTED = ใช้ไปแล้ว · REVOKED = ถูกแทนที่ด้วยคำเชิญใหม่ · PENDING = ยังใช้ได้")

print("\nเข้าสู่ระบบต่อได้ที่", APP_URL, "ด้วย", INVITE_EMAIL, "/", PROFILE["password"])

---

## หาลิงก์และ OTP จาก log (เมื่อ checkout ไม่ได้ส่งอีเมลจริง)

เมื่อ `SMTP_USER` ว่าง mailer จะพิมพ์เนื้ออีเมล ลิงก์ และ OTP ลง stdout แทนการส่ง

In [ ]:
import subprocess

logs = subprocess.run(
    ["docker", "compose", "logs", "backend", "--tail", "200"],
    cwd=CHECKOUT, capture_output=True, text=True,
).stdout

for line in logs.splitlines():
    if "register?token=" in line or "รหัส OTP" in line or INVITE_EMAIL in line:
        print(line)

## เริ่มรอบใหม่

เปลี่ยน `INVITE_EMAIL` เป็น `+demo2`, `+demo3` … แล้วรันตั้งแต่เซลล์ที่ 1 ใหม่
ถ้าต้องการยกเลิกคำเชิญที่ยังค้างโดยไม่ออกใบใหม่ ใช้เซลล์ล่างนี้

In [ ]:
_, listing = call("GET", "/api/admin/invitations", headers={"x-admin-token": ADMIN_TOKEN})
pending = [i for i in listing["invitations"] if i["email"] == INVITE_EMAIL and i["status"] == "PENDING"]

for item in pending:
    call("POST", f"/api/admin/invitations/{item['id']}/revoke", headers={"x-admin-token": ADMIN_TOKEN})

if not pending:
    print("ไม่มีคำเชิญที่ยังค้างของอีเมลนี้")